# 🎮 Unidad 5 — Unity ML-Agents: RL en Entornos 3D

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### Un notebook diferente a todos los anteriores

En este notebook **no escribiremos código Python de RL**. En cambio:
- Configuraremos ML-Agents, la herramienta de RL de Unity
- Analizaremos archivos YAML de hiperparámetros
- Lanzaremos entrenamientos con comandos de terminal
- Observaremos y analizaremos el comportamiento de agentes en 3D

Entrenaremos dos agentes con desafíos muy distintos:

| Entorno | Tarea | Dificultad | Por qué es interesante |
|---------|-------|------------|----------------------|
| 🐻 **SnowballTarget** | Disparar bolas de nieve a objetivos | ⭐⭐ | Recompensas frecuentes, buena introducción a ML-Agents |
| 🏛️ **Pirámides con RND** | Resolver una secuencia larga sin recompensas intermedias | ⭐⭐⭐⭐ | Introduce curiosidad intrínseca (RND) |

> ⚠️ **Este notebook tiene la instalación más compleja del curso.**  
> El setup de ML-Agents requiere ajustar la versión de Python de Colab.  
> Los errores en el setup son normales — esta sección explica cómo resolverlos.

> ⏱️ **Tiempo estimado:** ~3 horas (SnowballTarget: ~30 min · Pirámides: ~45 min · Setup: ~15 min)

---
## 1 · ¿Cómo funciona ML-Agents? Antes de instalar nada

### La arquitectura que usaremos

En los notebooks anteriores, Python controlaba todo: el entorno, el agente, el entrenamiento.
En ML-Agents el esquema es diferente:

```
┌─────────────────────────────────┐     ┌──────────────────────────────────┐
│   UNITY (ejecutable compilado)  │     │   PYTHON (tu Colab)              │
│                                 │     │                                  │
│  - Motor de física 3D           │◄───►│  - mlagents-learn                │
│  - Lógica del juego             │     │  - Algoritmo PPO/SAC             │
│  - Observaciones (raycasts)     │     │  - Actualizar los pesos θ        │
│  - Sistema de recompensas       │     │  - Guardar checkpoints           │
│  - Múltiples agentes en paralelo│     │  - Publicar en el Hub            │
└─────────────────────────────────┘     └──────────────────────────────────┘
         ↑                                          ↑
   Ejecutable Linux                      El comando que lanzamos
   (.x86_64, ya compilado)               con mlagents-learn
```

### Lo que NO necesitamos

- ❌ Instalar Unity (el ejecutable ya está compilado para Linux)
- ❌ Saber C# (el lenguaje de Unity)
- ❌ Escribir código de RL (ML-Agents incluye PPO y SAC implementados)

### Lo que SÍ necesitamos

- ✅ Clonar ML-Agents (la librería Python que se comunica con el ejecutable)
- ✅ Configurar el YAML de hiperparámetros
- ✅ Lanzar el comando `mlagents-learn`
- ✅ Analizar los logs y los resultados

---
## 2 · Configurar el entorno — el paso más crítico

### 2.1 · GPU obligatoria

ML-Agents con entornos 3D requiere GPU para entrenar en tiempos razonables:

| Configuración | SnowballTarget (200k pasos) | Pirámides (1M pasos) |
|--------------|----------------------------|---------------------|
| Sin GPU (CPU) | ~3–4 horas | ~8+ horas |
| GPU T4 | ~15–30 min | ~30–45 min |

**Activar GPU:** Entorno de ejecución → Cambiar tipo de entorno → GPU T4

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU detectada: {result.stdout.strip()}")
else:
    print("❌ No se detectó GPU — actívala antes de continuar")

### 2.2 · Paso 1: Clonar ML-Agents

Clonamos el repositorio oficial de ML-Agents desde GitHub.
La bandera `--depth 1` descarga solo el estado actual (sin historial),
reduciendo el tiempo de ~15 min a ~3 min.

In [ ]:
%%capture
# Clonar ML-Agents (puede tardar ~3 min)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents
print("✅ ML-Agents clonado")

### 2.3 · Paso 2: Ajustar la versión de Python

ML-Agents requiere Python 3.10.x. Colab a veces usa una versión diferente.
Instalamos Miniconda para crear un entorno con la versión correcta.

**Errores comunes en este paso:**

| Error | Causa | Solución |
|-------|-------|---------|
| `ModuleNotFoundError: mlagents` | Python incorrecto | Re-ejecutar esta celda |
| `pip: command not found` | Entorno no activado | Reiniciar el entorno de Colab y repetir |
| Celda tarda más de 5 min | Descarga lenta | Normal, esperar |

In [ ]:
# Ver la versión actual de Python en Colab
!python --version

In [ ]:
%%capture
# Instalar Miniconda y crear entorno con Python 3.10.12
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
!conda install -q -y --prefix /usr/local python=3.10.12

# Verificar la versión instalada
!python --version

### 2.4 · Paso 3: Instalar ML-Agents desde el código fuente

Instalamos ML-Agents desde el repositorio que acabamos de clonar.
Esto garantiza compatibilidad entre la librería Python y el ejecutable Unity.

In [ ]:
%%capture
%cd ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

### 2.5 · Verificar que todo está listo

In [ ]:
# Verificar que mlagents-learn está disponible
result = subprocess.run(['mlagents-learn', '--help'],
                        capture_output=True, text=True)
if result.returncode == 0 or 'usage' in result.stdout.lower() or 'usage' in result.stderr.lower():
    print("✅ mlagents-learn está instalado y listo")
    # Mostrar versión
    !mlagents-learn --version 2>/dev/null || echo '(versión no disponible con --version)'
else:
    print("❌ mlagents-learn no encontrado")
    print("   Solución: ejecuta de nuevo las celdas 2.3 y 2.4, luego reinicia el entorno")

---
## 3 · Parte 1 — SnowballTarget 🐻❄️

### 3.1 · Descargar el ejecutable de Unity

El entorno de SnowballTarget es un ejecutable compilado para Linux (Ubuntu).
Ocupa ~50 MB y contiene el motor de juego completo — no necesitamos Unity instalado.

Después de descargar, damos permisos de ejecución con `chmod` —
sin esto Linux no permite ejecutar el binario.

In [ ]:
import os
os.makedirs('./training-envs-executables/linux/', exist_ok=True)
print("✅ Directorio creado")

In [ ]:
# Descargar SnowballTarget (~50 MB)
!wget -q 'https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip' \
     -O ./training-envs-executables/linux/SnowballTarget.zip
print("✅ Descargado")

In [ ]:
%%capture
# Descomprimir
!unzip -d ./training-envs-executables/linux/ \
       ./training-envs-executables/linux/SnowballTarget.zip

# Dar permisos de ejecución
!chmod -R 755 ./training-envs-executables/linux/SnowballTarget
print("✅ Ejecutable listo")

In [ ]:
# Verificar que el ejecutable está en su lugar
ruta = './training-envs-executables/linux/SnowballTarget'
if os.path.exists(ruta):
    archivos = os.listdir(ruta)
    print(f"✅ Ejecutable disponible. Archivos: {archivos}")
else:
    print("❌ No se encontró el ejecutable. Re-ejecuta las celdas anteriores.")

### 3.2 · El archivo YAML de SnowballTarget — conectando teoría y práctica

A diferencia de RL-Zoo (Unit 3), en ML-Agents escribimos el YAML nosotros.
Aquí está el archivo completo con cada parámetro explicado:

| Parámetro YAML | Valor | Conexión con la teoría |
|----------------|-------|----------------------|
| `trainer_type: ppo` | PPO | Algoritmo que estudiaremos en el Módulo 8 — el mismo de LunarLander |
| `max_steps: 200000` | 200k pasos | Total de pasos de entrenamiento (ajustado para Colab) |
| `time_horizon: 64` | 64 pasos | Cada cuántos pasos actualiza la política aunque no haya terminado el episodio |
| `batch_size: 128` | 128 | Tamaño del mini-lote para el gradiente |
| `buffer_size: 2048` | 2048 | Pasos acumulados antes de cada actualización (como `n_steps` en SB3) |
| `learning_rate: 0.0003` | 3×10⁻⁴ | α — igual al valor típico de PPO |
| `hidden_units: 256` | 256 | Neuronas en las capas ocultas de la red |
| `num_layers: 2` | 2 capas | Profundidad de la red neuronal |
| `gamma: 0.995` | 0.995 | Factor de descuento — casi sin descuento |
| `keep_checkpoints: 10` | 10 | Cuántos checkpoints guardar (los últimos 10) |
| `summary_freq: 10000` | cada 10k | Cada cuántos pasos imprime estadísticas |

In [ ]:
# Crear el directorio de configuración si no existe
os.makedirs('./config/ppo', exist_ok=True)

# Escribir el archivo YAML de SnowballTarget
snowball_yaml = '''behaviors:
  SnowballTarget:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    max_steps: 200000
    time_horizon: 64
    threaded: true
    hyperparameters:
      learning_rate: 0.0003
      batch_size: 128
      buffer_size: 2048
      learning_rate_schedule: linear
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
'''

with open('./config/ppo/SnowballTarget.yaml', 'w') as f:
    f.write(snowball_yaml)

print("✅ Archivo SnowballTarget.yaml creado en ./config/ppo/")
print("\nContenido:")
!cat ./config/ppo/SnowballTarget.yaml

### 3.3 · Lanzar el entrenamiento

El comando `mlagents-learn` conecta Python con el ejecutable Unity y lanza el entrenamiento.
Desglose de cada flag:

```bash
mlagents-learn
    ./config/ppo/SnowballTarget.yaml    # ← archivo YAML con hiperparámetros
    --env=./training-envs-executables/linux/SnowballTarget/SnowballTarget
                                        # ← ruta al ejecutable de Unity
    --run-id='SnowballTarget1'          # ← nombre del experimento (para los logs)
    --no-graphics                       # ← sin ventana gráfica (Colab no tiene pantalla)
```

> ☕ Este entrenamiento tarda **~15–30 minutos con GPU**.  
> El agente empieza disparando en direcciones aleatorias — hacia el final aprende a apuntar.

In [ ]:
!mlagents-learn ./config/ppo/SnowballTarget.yaml \
    --env=./training-envs-executables/linux/SnowballTarget/SnowballTarget \
    --run-id='SnowballTarget1' \
    --no-graphics

### 3.4 · Leer los logs de ML-Agents

Durante el entrenamiento ML-Agents imprime estadísticas cada `summary_freq` pasos.
Son diferentes a los logs de SB3 — aquí la guía de lectura:

```
[INFO] SnowballTarget. Step: 10000. 
  Mean Reward: 2.310   ← PUNTUACIÓN MEDIA — lo más importante, debe subir
  Std of Reward: 1.400 ← variabilidad — alta al inicio, baja cuando el agente es consistente
  Training.                  
[INFO] SnowballTarget. Step: 50000.
  Mean Reward: 7.850   ← subiendo
  Std of Reward: 2.100
[INFO] SnowballTarget. Step: 200000.
  Mean Reward: 16.500  ← cerca de la meta (≥15)
  Std of Reward: 1.200 ← más consistente
```

| Métrica | Al inicio | Cuando aprende | Señal de alarma |
|---------|-----------|----------------|-----------------|
| `Mean Reward` | ~0–2 | Sube gradualmente | No sube después de 100k pasos |
| `Std of Reward` | Alta (>3) | Baja (<2) | Sube en lugar de bajar |

> 🎯 **Meta de certificación HF:** Mean Reward ≥ 15 objetivos por episodio.

In [ ]:
# Ver los resultados guardados después del entrenamiento
resultado_dir = './results/SnowballTarget1'
if os.path.exists(resultado_dir):
    print("Archivos generados por el entrenamiento:")
    for root, dirs, files in os.walk(resultado_dir):
        for f in files:
            ruta = os.path.join(root, f)
            size = os.path.getsize(ruta)
            print(f"  {ruta}  ({size:,} bytes)")
else:
    print("El directorio de resultados no existe aún.")
    print("Ejecuta la celda de entrenamiento primero.")

### 3.5 · Comparar checkpoints — evolución del aprendizaje

ML-Agents guarda hasta `keep_checkpoints` versiones del modelo durante el entrenamiento.
Puedes comparar el comportamiento del agente en distintos momentos:

| Checkpoint | Comportamiento típico |
|-----------|----------------------|
| 20k pasos | Dispara en todas direcciones, casi nunca acierta |
| 80k pasos | Empieza a orientarse hacia el objetivo antes de disparar |
| 150k pasos | Aprende a minimizar el tiempo entre disparos |
| 200k pasos | Política estable: apunta, dispara, se reorienta rápido |

El modelo final (`.onnx`) es el que se sube al Hub y se puede ejecutar en el navegador.

---

# 🔒 Sección opcional — Publicar SnowballTarget en el Hub

> Requiere cuenta de HuggingFace. La meta de certificación es Mean Reward ≥ 15.

---

## [OPCIONAL] Publicar SnowballTarget en el Hub 🤗

### Paso 1: Iniciar sesión

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Paso 2: Subir el modelo con mlagents-push-to-hf

Este comando evalúa el modelo, genera el video replay y sube todo al Hub:

- `--run-id`: debe coincidir con el `--run-id` que usaste al entrenar
- `--local-dir`: carpeta donde ML-Agents guardó los resultados
- `--repo-id`: tu nombre de usuario y el nombre del repositorio en el Hub
- `--commit-message`: descripción del modelo

In [ ]:
USERNAME = ''   # ← pon tu usuario de HuggingFace (distingue mayúsculas)

!mlagents-push-to-hf \
    --run-id='SnowballTarget1' \
    --local-dir='./results/SnowballTarget1' \
    --repo-id='{USERNAME}/ppo-SnowballTarget' \
    --commit-message='Mi oso entrenado 🐻❄️'

### Paso 3: Ver tu agente en el navegador

Una vez publicado:
1. Ve a: **https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget**
2. En *Step 1*: escribe tu nombre de usuario (exactamente igual, distingue mayúsculas)
3. En *Step 2*: selecciona tu repositorio `ppo-SnowballTarget`
4. Haz clic en **Load model** y disfruta viendo al oso jugar en tu navegador 🎮

---
## 4 · Parte 2 — Pirámides con Curiosidad RND 🏛️

### 4.1 · Por qué Pirámides necesita RND y SnowballTarget no

Antes de descargar nada, entendamos la diferencia fundamental entre estos dos entornos:

| Característica | SnowballTarget | Pirámides |
|----------------|---------------|-----------|
| Recompensa | +1 por cada objetivo derribado | Solo +1 al llegar al oro al final |
| Frecuencia de recompensa | **Alta** — cada pocos segundos | **Muy baja** — solo al completar la secuencia |
| Secuencia necesaria | Solo apuntar y disparar | Botón → derribar pirámide → recoger oro |
| ¿Puede PPO aprender solo? | ✅ Sí — hay señal de recompensa constante | ❌ No — sin RND el agente nunca descubre la secuencia |

El problema de Pirámides se llama **recompensa escasa** (*sparse reward* en inglés):
si el agente nunca completa la secuencia por casualidad, nunca recibe recompensa,
y sin recompensa no hay gradiente, y sin gradiente no aprende.

**RND** (Random Network Distillation) resuelve esto añadiendo una recompensa interna
por visitar estados nuevos — el agente explora aunque no haya recompensa externa.

### 4.2 · Descargar el ejecutable de Pirámides

In [ ]:
# Descargar Pirámides (~150 MB)
!wget -q 'https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip' \
     -O ./training-envs-executables/linux/Pyramids.zip
print("✅ Descargado")

In [ ]:
%%capture
!unzip -d ./training-envs-executables/linux/ \
       ./training-envs-executables/linux/Pyramids.zip
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids
print("✅ Ejecutable de Pirámides listo")

### 4.3 · El YAML de PyramidsRND — la sección `rnd` explicada

A diferencia de SnowballTarget, `PyramidsRND.yaml` ya viene incluido en el repositorio
de ML-Agents. Solo necesitamos hacer **un cambio**: reducir `max_steps` de 10,000,000
a 1,000,000 para que el entrenamiento sea manejable en Colab.

Pero lo más interesante es la sección `reward_signals`, que ahora tiene **dos entradas**:

In [ ]:
# Ver el contenido original del YAML de Pirámides
!cat ./config/ppo/PyramidsRND.yaml 2>/dev/null || echo 'Archivo no encontrado, buscando...'
!find . -name 'PyramidsRND.yaml' 2>/dev/null | head -5

### Las dos señales de recompensa

```yaml
reward_signals:
  extrinsic:          # ← recompensa real del entorno
    gamma: 0.99
    strength: 1.0     # peso de la recompensa externa
  rnd:                # ← recompensa de curiosidad (RND)
    gamma: 0.99
    strength: 0.5     # β — cuánto peso tiene la curiosidad
    encoding_size: 64
    learning_rate: 1e-4
```

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `extrinsic.strength` | 1.0 | La recompensa real vale el 100% |
| `rnd.strength` (β) | 0.5 | La curiosidad vale el 50% — β del Cap. 5.3 |
| `rnd.encoding_size` | 64 | Tamaño del espacio de representación de RND |

**La recompensa total** en cada paso es:
$$R_{\text{total}} = R_{\text{extrínseca}} \times 1.0 + R_{\text{RND}} \times 0.5$$

Con el tiempo, la red predictora de RND aprende a predecir la red objetivo,
el error baja, y `R_RND` se acerca a cero — dejando solo la recompensa real.

In [ ]:
# Modificar max_steps de 10M a 1M (el único cambio necesario)
import re

yaml_path = './config/ppo/PyramidsRND.yaml'

# Intentar leer el archivo existente
try:
    with open(yaml_path, 'r') as f:
        contenido = f.read()
    # Cambiar max_steps
    contenido_mod = re.sub(r'max_steps:\s*\d+', 'max_steps: 1000000', contenido)
    with open(yaml_path, 'w') as f:
        f.write(contenido_mod)
    print("✅ max_steps cambiado a 1,000,000")
except FileNotFoundError:
    print("⚠️ Archivo no encontrado. Creando PyramidsRND.yaml desde cero...")
    pyramids_yaml = '''behaviors:
  Pyramids:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    max_steps: 1000000
    time_horizon: 128
    threaded: true
    hyperparameters:
      learning_rate: 0.0003
      batch_size: 128
      buffer_size: 2048
      learning_rate_schedule: linear
      beta: 0.001
      epsilon: 0.2
      lambd: 0.99
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
      rnd:
        gamma: 0.99
        strength: 0.5
        encoding_size: 64
        learning_rate: 1.0e-4
'''
    with open(yaml_path, 'w') as f:
        f.write(pyramids_yaml)
    print("✅ PyramidsRND.yaml creado con max_steps: 1,000,000")

### 4.4 · Lanzar el entrenamiento de Pirámides

> ⏱️ Este entrenamiento tarda **~30–45 minutos con GPU T4**.
> Es significativamente más largo que SnowballTarget porque el espacio de exploración
> es más grande y RND necesita tiempo para generar señal útil.

> 📌 **Consejo:** monta Google Drive antes de entrenar para no perder el progreso
> si Colab se desconecta:

In [ ]:
# Montar Google Drive (recomendado para entrenamientos largos)
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('/content/drive/MyDrive/rl-course/unit5', exist_ok=True)
print("Descomenta las líneas de arriba si quieres guardar en Google Drive")

In [ ]:
!mlagents-learn ./config/ppo/PyramidsRND.yaml \
    --env=./training-envs-executables/linux/Pyramids/Pyramids \
    --run-id='Pyramids-Training' \
    --no-graphics

### 4.5 · Analizar los logs — la señal RND en acción

Los logs de Pirámides muestran **dos líneas de recompensa** — una por cada señal.
Esta es la clave para entender si RND está funcionando correctamente:

```
[INFO] Pyramids. Step: 50000.
  Mean Reward: 0.050        ← recompensa extrínseca (casi cero — raro encontrar el oro)
  Std of Reward: 0.200
  Extrinsic Reward: 0.050   ← misma que Mean Reward al inicio
  Curiosity Reward: 2.340   ← RND alta — el agente explora estados nuevos constantemente

[INFO] Pyramids. Step: 500000.
  Mean Reward: 0.820        ← subiendo — el agente empieza a completar la secuencia
  Extrinsic Reward: 0.820
  Curiosity Reward: 0.450   ← RND bajando — el agente ya conoce el entorno

[INFO] Pyramids. Step: 1000000.
  Mean Reward: 1.650        ← cerca de la meta (≥1.75)
  Extrinsic Reward: 1.650
  Curiosity Reward: 0.120   ← RND muy baja — la política extrínseca tomó el control
```

**La dinámica que observarás:**
- Al inicio: RND alta, recompensa extrínseca casi cero
- A mitad: RND bajando, extrínseca subiendo — el agente encontró la secuencia
- Al final: RND casi cero, extrínseca dominante — solo maximiza la recompensa real

> 🎯 **Meta de certificación HF:** Mean Reward ≥ 1.75 (sobre un máximo de ~2.0)

---

# 🔒 Sección opcional — Publicar Pirámides en el Hub

> Requiere cuenta de HuggingFace. Meta: Mean Reward ≥ 1.75

---

## [OPCIONAL] Publicar Pirámides en el Hub 🤗

In [ ]:
# Si no iniciaste sesión todavía:
# from huggingface_hub import notebook_login
# notebook_login()

USERNAME = ''   # ← pon tu usuario de HuggingFace

!mlagents-push-to-hf \
    --run-id='Pyramids-Training' \
    --local-dir='./results/Pyramids-Training' \
    --repo-id='{USERNAME}/ppo-Pyramids' \
    --commit-message='Agente Pirámides con curiosidad RND 🏛️'

### Ver tu agente jugando

Una vez publicado:
1. Ve a: **https://huggingface.co/spaces/unity/ML-Agents-Pyramids**
2. Escribe tu nombre de usuario y selecciona tu repositorio
3. Observa cómo el agente presiona el botón, derriba la pirámide y recoge el oro — todo sin que nadie le programara esa secuencia

---
## 5 · 🏋️ Ejercicio: entrena en otro entorno de ML-Agents

### El ejercicio de este notebook

Ya completaste el pipeline completo con dos entornos muy distintos.
Ahora aplícalo a un tercer entorno de tu elección.

**Entornos oficiales de ML-Agents disponibles:**

| Entorno | Tarea | Dificultad | Algoritmo sugerido |
|---------|-------|------------|-------------------|
| **Walker** | Robot bípedo aprende a caminar | ⭐⭐⭐ | SAC (acciones continuas) |
| **Crawler** | Criatura de 4 brazos aprende a moverse | ⭐⭐⭐ | SAC |
| **Worm** | Gusano aprende a desplazarse | ⭐⭐⭐ | SAC |
| **Hallway** | Agente que memoriza señales visuales | ⭐⭐⭐⭐ | PPO con memoria (LSTM) |
| **3DBall** | Equilibrar una pelota en una plataforma | ⭐⭐ | PPO |
| **FoodCollector** | Recoger comida evitando obstáculos | ⭐⭐ | PPO |

**Repositorio completo:** https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Learning-Environment-Examples.md

**Pasos:**
1. Elige un entorno de la tabla
2. Busca su ejecutable Linux en el repositorio de ML-Agents
3. Encuentra o crea su archivo YAML de configuración
4. Lanza el entrenamiento con el mismo comando `mlagents-learn`
5. Observa y documenta: ¿qué comportamiento emergió? ¿Necesitó curiosidad?

**Preguntas de reflexión:**
- ¿El entorno tiene recompensas escasas o densas?
- ¿Usarías PPO o SAC? ¿Por qué?
- ¿Qué comportamientos emergieron sin ser programados explícitamente?

In [ ]:
# Espacio para tus notas del experimento
mi_experimento = {
    'entorno': '',          # nombre del entorno elegido
    'run_id': '',           # nombre del experimento
    'max_steps': 0,         # pasos usados
    'mean_reward_final': 0, # recompensa media al final
    'observaciones': '',    # ¿qué comportamiento aprendió?
    'necesito_rnd': None,   # True/False — ¿por qué?
}
print("Rellena el diccionario con los resultados de tu experimento.")
print("Luego comparte con tus compañeros: ¿cuál encontró el entorno más interesante?")

---

## 🎉 ¡Notebook completado! El Módulo 5 está completo.

Trabajaste con el pipeline de ML-Agents de principio a fin.
Lo que dominaste:

1. **Arquitectura de ML-Agents** — ejecutable Unity + sockets + Python
2. **Setup complejo** — Miniconda, Python 3.10, instalación desde source
3. **YAML de hiperparámetros** — conectando cada parámetro con la teoría
4. **Lectura de logs** — interpretar `Mean Reward`, `Std of Reward`, `Curiosity Reward`
5. **SnowballTarget** — pipeline completo con recompensas densas
6. **Pirámides + RND** — curiosidad intrínseca para recompensas escasas
7. **La dinámica RND** — curiosity alta al inicio, extrínseca toma el control al final

### ¿Qué sigue en el curso de HuggingFace?

- **Módulo 6:** Actor-Critic (A2C) — combinar métodos de política y de valor
- **Módulo 7:** Multi-Agent RL — varios agentes aprendiendo juntos (SoccerTwos)
- **Módulo 8:** PPO completo desde cero con CleanRL
- **Bonus:** Decision Transformers, RLHF, RL con modelos de lenguaje

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*